In [16]:
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [17]:
import catboost

**Первый вариант датасета по керну 09.2026**

In [3]:
df = pd.read_csv("Data/data_kern.csv")

**Второй вариант датасета по керну 05.10.2026**

In [18]:
df = pd.read_csv("Data/data_kern_05_10.csv")

In [19]:
df

,Calibration,Well,Sample,Depth,Stratigraphy,Mg,Al,Si,P,S,...,Depth.2,Mo/Al,Depth.3,Sr/Ca,Depth.4,Ni/Si+Ca+Mg,Depth.5,Se/Ti,Depth.6,Si\Al
0,LE-FP,C3,1,2868.11,Fr,1.821148,7.515664,18.611483,0.299690,10.465318,...,2868.11,0.000849,2868.11,0.011767,2868.11,0.000142,2868.11,0.002077,2868.11,2.476359
1,LE-FP,C3,2,2868.30,Fr,1.266357,7.461183,16.904798,0.000000,14.653735,...,2868.30,0.000961,2868.30,NaN,2868.30,0.000161,2868.30,0.002284,2868.30,2.265699
2,LE-FP,C3,3,2868.45,Fr,1.724880,7.150965,18.068731,0.000000,13.442425,...,2868.45,0.000845,2868.45,0.332288,2868.45,0.000054,2868.45,0.002578,2868.45,2.526754
3,LE-FP,C3,4,2868.64,Fr,1.646439,8.260792,21.260196,0.000000,10.257598,...,2868.64,0.000659,2868.64,NaN,2868.64,0.000163,2868.64,0.000981,2868.64,2.573627
4,LE-FP,C3,5,2868.78,Fr,1.847176,7.757235,19.837340,0.000000,10.623499,...,2868.78,0.000840,2868.78,0.033692,2868.78,0.000140,2868.78,0.004081,2868.78,2.557270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3040,LE-FP,C7,362,2697.50,Tum,1.583924,10.335025,27.434503,0.029277,0.622963,...,2697.50,0.000000,2697.50,0.230056,2697.50,0.000000,2697.50,0.000000,2697.50,2.654517
3041,LE-FP,C7,363,2697.78,Tum,1.491458,7.860884,29.874495,0.020000,0.691707,...,2697.78,0.000000,2697.78,0.009905,2697.78,0.000000,2697.78,0.000000,2697.78,3.800399
3042,LE-FP,C7,364,2697.84,Tum,1.459012,11.210366,28.113407,0.052834,0.137574,...,2697.84,0.000000,2697.84,0.439778,2697.84,0.000000,2697.84,0.000000,2697.84,2.507805
3043,LE-FP,C7,365,2698.97,Tum,1.300348,1.489742,2.630132,0.000000,31.978827,...,2698.97,0.000000,2698.97,NaN,2698.97,0.001189,2698.97,0.000000,2698.97,1.765494


**Отобранные элементы с доставерными определениями**

In [20]:
elements = ['Mg', 'Al', 'Si', 'P', 'S', 'K', 'Ca', 'Ti', 'Mn', 'Fe', 'Ni', 'Cu', 'Zn', 'Se', 'Rb', 'Sr', 'Zr', 'Mo']

In [21]:
cols_to_keep = elements + ['Well', 'Sample',	'Depth',	'Stratigraphy']
df_filtered = df[cols_to_keep].query("Stratigraphy != 'Tum'").copy()

**Замена пачек на цифровые коды**

In [22]:
def to_str_key(val):
    try:
        return str(int(val))
    except (ValueError, TypeError):
        return str(val).strip()

df_filtered['Strat'] = df_filtered['Stratigraphy'].apply(to_str_key)

In [23]:
formation_map = {
    'Abl': 0,
    'A0': 1,
    '1': 2,
    '2a': 3,
    '2b': 4,
    '3': 5,
    '4a': 6,
    '4b': 7,
    '5a': 8,
    '5b': 9,
    '6': 10,
    'Fr': 11
}

df_filtered['Strat'] = df_filtered['Strat'].map(formation_map)

**Подсчет нулей по всем признакам**

In [24]:
# Подсчет нулей по всем признакам
zero_counts = df_filtered[elements].isnull().sum() + (df_filtered[elements] == 0).sum()

# Выводим результат
print("Количество нулевых значений по каждому элементу:")
print(zero_counts)

# Общее количество нулей
total_zeros = zero_counts.sum()
print(f"\nОбщее количество нулевых значений: {total_zeros}")

# Процент нулевых значений
total_values = len(df_filtered) * len(elements)
zero_percentage = (total_zeros / total_values) * 100
print(f"Процент нулевых значений: {zero_percentage:.2f}%")

Количество нулевых значений по каждому элементу:
Mg     344
Al       0
Si       3
P     1321
S        5
K      141
Ca     440
Ti       2
Mn     449
Fe     183
Ni     215
Cu     348
Zn      23
Se     346
Rb     257
Sr      93
Zr      26
Mo     591
dtype: int64

Общее количество нулевых значений: 4787
Процент нулевых значений: 8.78%


**Подсчет значений ниже LOD**

In [ ]:
# LOD для каждого элемента
lod = {
    'Mg': 0.1848625,
    'Al': 0.0927875,
    'Si': 0.2984125,
    'Ca': 0.077025,
    'S': 0.3082,
    'K': 0.0271375,
    'Ti': 0.003,
    'Mn': 0.0024375,
    'Fe': 0.09165,
    'Ni': 0.0032625,
    'Cu': 0.00335,
    'Zn': 0.0022375,
    'Rb': 0.0003125,
    'Sr': 0.0013125,
    'Zr': 0.00065,
    'Mo': 0.00105,
    'P': 0.016325,
    'Se': 0.0002375
}


In [25]:
below_lod = pd.Series({e: ((df_filtered[e] > 0) & (df_filtered[e] < lod[e])).sum() for e in elements})

print("Количество значений ниже LOD (без нулей):")
print(below_lod)

print(f"\nВсего ниже LOD: {below_lod.sum()}")
print(f"Процент: {(below_lod.sum() / (len(df_filtered) * len(elements))) * 100:.2f}%")


Количество значений ниже LOD (без нулей):
Mg      3
Al      3
Si      2
P     480
S      20
K     120
Ca    127
Ti      3
Mn     72
Fe     44
Ni    490
Cu     99
Zn      0
Se    494
Rb      0
Sr      0
Zr     34
Mo    246
dtype: int64

Всего ниже LOD: 2237
Процент: 4.10%


**Поиск минимальных значений по элементам**

In [26]:
min_values = pd.Series({e: df_filtered.loc[df_filtered[e] > 0, e].min() for e in elements})

print("Минимальные значения (исключая 0):")
print(min_values)

Минимальные значения (исключая 0):
Mg    0.130200
Al    0.013244
Si    0.026589
P     0.000358
S     0.075067
K     0.000100
Ca    0.000877
Ti    0.000446
Mn    0.001031
Fe    0.002603
Ni    0.000112
Cu    0.002408
Zn    0.002500
Se    0.000005
Rb    0.000489
Sr    0.005073
Zr    0.000092
Mo    0.000663
dtype: float64


**Минимальное или LOD, если он меньше**

In [27]:
result = pd.Series({e: min(min_values[e], lod[e]) for e in elements})

print("Минимум (меньше из минимального значения и LOD):")
print(result)

Минимум (меньше из минимального значения и LOD):
Mg    0.130200
Al    0.013244
Si    0.026589
P     0.000358
S     0.075067
K     0.000100
Ca    0.000877
Ti    0.000446
Mn    0.001031
Fe    0.002603
Ni    0.000112
Cu    0.002408
Zn    0.002237
Se    0.000005
Rb    0.000313
Sr    0.001313
Zr    0.000092
Mo    0.000663
dtype: float64


**Замена нулей на 2/3 LOD (result)**

In [28]:
# Замена нулей на 2/3 LOD
for el in elements:
    replacement = (2/3) * result[el]
    df_filtered[el] = df_filtered[el].replace(0, replacement)

# Проверка
print("Нулей после замены:")
print((df_filtered[elements] == 0).sum())

Нулей после замены:
Mg    0
Al    0
Si    0
P     0
S     0
K     0
Ca    0
Ti    0
Mn    0
Fe    0
Ni    0
Cu    0
Zn    0
Se    0
Rb    0
Sr    0
Zr    0
Mo    0
dtype: int64


**Создание новых признаков по соотношениям элементов**

In [29]:
# Карбонатность
df_filtered['(Ca+Mg)/(Al+Si)'] = (df_filtered['Ca'] + df_filtered['Mg']) / (df_filtered['Al'] + df_filtered['Si'])
df_filtered['Sr/Ca'] = df_filtered['Sr'] / df_filtered['Ca']
df_filtered['Mn/Ca'] = df_filtered['Mn'] / df_filtered['Ca']

# Терригенное вещество
df_filtered['(Al+K)/(Ca+Mg+Si)'] = (df_filtered['Al'] + df_filtered['K']) / (df_filtered['Ca'] + df_filtered['Mg'] + df_filtered['Si'])
df_filtered['Zr/Al'] = df_filtered['Zr'] / df_filtered['Al']
df_filtered['Zr/Si'] = df_filtered['Zr'] / df_filtered['Si']
df_filtered['K/Al'] = df_filtered['K'] / df_filtered['Al']
df_filtered['Ti/(Zr+Ti)'] = df_filtered['Ti'] / (df_filtered['Zr'] + df_filtered['Ti'])

# Морская биопродуктивность и кислородный режим
df_filtered['Si/Al'] = df_filtered['Si'] / df_filtered['Al']
df_filtered['P/Al'] = df_filtered['P'] / df_filtered['Al']
df_filtered['Ni/Fe'] = df_filtered['Ni'] / df_filtered['Fe']
df_filtered['Ni/(Si+Ca+Mg)'] = df_filtered['Ni'] / (df_filtered['Si'] + df_filtered['Ca'] + df_filtered['Mg'])
df_filtered['S/(Rb+Zr)'] = df_filtered['S'] / (df_filtered['Rb'] + df_filtered['Zr'])
df_filtered['Mo/Al'] = df_filtered['Mo'] / df_filtered['Al']
df_filtered['Mo/(Mn+Mo)'] = df_filtered['Mo'] / (df_filtered['Mn'] + df_filtered['Mo'])
df_filtered['Mn/Al'] = df_filtered['Mn'] / df_filtered['Al']

# Сульфидность
df_filtered['Fe/S'] = df_filtered['Fe'] / df_filtered['S']
df_filtered['Ni/S'] = df_filtered['Ni'] / df_filtered['S']

# Железистость
df_filtered['Fe/(Ca+Mg+Al)'] = df_filtered['Fe'] / (df_filtered['Ca'] + df_filtered['Mg'] + df_filtered['Al'])
df_filtered['Ti/(Si+Ca+Mg)'] = df_filtered['Ti'] / (df_filtered['Si'] + df_filtered['Ca'] + df_filtered['Mg'])
df_filtered['Fe/(Mn+Fe)'] = df_filtered['Fe'] / (df_filtered['Mn'] + df_filtered['Fe'])

# Прочие
df_filtered['(Al+K+Ti)/Si'] = (df_filtered['Al'] + df_filtered['K'] + df_filtered['Ti']) / df_filtered['Si']
df_filtered['(Al+K+Ti)/S'] = (df_filtered['Al'] + df_filtered['K'] + df_filtered['Ti']) / df_filtered['S']
df_filtered['(Ti+Rb+Zr)/S'] = (df_filtered['Ti'] + df_filtered['Rb'] + df_filtered['Zr']) / df_filtered['S']
df_filtered['S/Ti'] = df_filtered['S'] / df_filtered['Ti']
df_filtered['Ni/Al'] = df_filtered['Ni'] / df_filtered['Al']
df_filtered['Zn/Al'] = df_filtered['Zn'] / df_filtered['Al']
df_filtered['Se/Si'] = df_filtered['Se'] / df_filtered['Si']
df_filtered['(Ca+Mg)/Al'] = (df_filtered['Ca'] + df_filtered['Mg']) / df_filtered['Al']
df_filtered['Se/Ti'] = df_filtered['Se'] / df_filtered['Ti']
df_filtered['(10*Rb+Ti+10*Zr)/Se*1000'] = (10 * df_filtered['Rb'] + df_filtered['Ti'] + 10 * df_filtered['Zr']) / df_filtered['Se'] * 1000

In [30]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3028 entries, 0 to 3037
Data columns (total 54 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Mg                        3028 non-null   float64
 1   Al                        3028 non-null   float64
 2   Si                        3028 non-null   float64
 3   P                         3028 non-null   float64
 4   S                         3028 non-null   float64
 5   K                         3028 non-null   float64
 6   Ca                        3028 non-null   float64
 7   Ti                        3028 non-null   float64
 8   Mn                        3028 non-null   float64
 9   Fe                        3028 non-null   float64
 10  Ni                        3028 non-null   float64
 11  Cu                        3028 non-null   float64
 12  Zn                        3028 non-null   float64
 13  Se                        3028 non-null   float64
 14  Rb           

In [31]:
df_filtered.to_csv('Data/data_kern_ratios_06_10.csv', index=False)